# Pulizia e Unificazione del Dataset — Beni Confiscati alla Mafia
Questo notebook carica, pulisce e unifica i quattro file CSV relativi ai beni confiscati alle organizzazioni criminali in Italia, pubblicati dall'**ANBSC** (Agenzia Nazionale per l'Amministrazione e la Destinazione dei Beni Sequestrati e Confiscati alla Criminalità Organizzata).

I quattro file sono:
- `Aziende destinate.csv` — aziende già assegnate con decreto
- `Aziende in gestione.csv` — aziende ancora in amministrazione giudiziaria
- `Immobili destinati.csv` — immobili già assegnati con decreto
- `Immobili in gestione.csv` — immobili ancora in gestione

Al termine, il dataset unificato viene arricchito con le **coordinate geografiche comunali** (via join con Wikidata, proprietà P625) ed esportato come CSV e datapackage Frictionless Data.

In [1]:
import pandas as pd
import numpy as np
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
from utils import printInfo, esporta_csv_e_frictionless

## 1. Caricamento dei file raw

Carichiamo i quattro CSV. I file *destinati* e *in gestione* hanno strutture leggermente diverse:
- **destinati**: contengono `regione`, `provincia`, `comune`, `indirizzo`, `decreto_*`, `destinatario`, `fine`, `utilizzo`
- **in gestione**: contengono `nome_regione`, `nome_provincia`, `nome_comune`, `provvedimento`, `quota_confiscata`, `tipologia_procedura`

Entrambi condividono: `s_bene`, `genere`, `categoria`, `sottocategoria`, `distretto`, `NomeComuneValidato`, `NomeProvinciaValidato`, `NomeRegioneValidato`, `CODISTAT`, `CODISTATPROV`, `CODISTATREG`.

In [2]:
PATH_RAW = '../dataset/raw/'

az_dest  = pd.read_csv(PATH_RAW + 'Aziende destinate.csv',    encoding='utf-8')
az_gest  = pd.read_csv(PATH_RAW + 'Aziende in gestione.csv',  encoding='utf-8')
imm_dest = pd.read_csv(PATH_RAW + 'Immobili destinati.csv',   encoding='utf-8')
imm_gest = pd.read_csv(PATH_RAW + 'Immobili in gestione.csv', encoding='utf-8')

print(f'Aziende destinate:    {len(az_dest):>6,} righe')
print(f'Aziende in gestione:  {len(az_gest):>6,} righe')
print(f'Immobili destinati:   {len(imm_dest):>6,} righe')
print(f'Immobili in gestione: {len(imm_gest):>6,} righe')
print(f'Totale:               {len(az_dest)+len(az_gest)+len(imm_dest)+len(imm_gest):>6,} righe')

Aziende destinate:       952 righe
Aziende in gestione:   3,022 righe
Immobili destinati:   14,874 righe
Immobili in gestione: 17,453 righe
Totale:               36,301 righe


## 2. Standardizzazione delle colonne

I file *in gestione* usano il prefisso `nome_` per regione/provincia/comune, mentre i file *destinati* no. Rinominiamo per uniformare, poi aggiungiamo due colonne descrittive:
- `tipo`: `'immobile'` o `'azienda'`
- `stato`: `'destinato'` o `'in_gestione'`

Queste due colonne sono il punto di partenza per tutte le analisi comparative.

In [3]:
COLS_COMUNI = [
    's_bene', 'genere', 'categoria', 'sottocategoria',
    'distretto', 'NomeComuneValidato', 'NomeProvinciaValidato',
    'NomeRegioneValidato', 'CODISTAT', 'CODISTATPROV', 'CODISTATREG'
]

COLS_DESTINATI = [
    'indirizzo', 'ufficio', 'procedura', 'tipologia',
    'destinatario', 'fine', 'utilizzo',
    'decreto_numero', 'decreto_anno', 'decreto_data'
]

COLS_GESTIONE = [
    'provvedimento', 'quota_confiscata',
    'ufficio_giudiziario', 'tipologia_procedura'
]

ALL_COLS = ['tipo', 'stato'] + COLS_COMUNI + COLS_DESTINATI + COLS_GESTIONE

def prepara_dataframe(df, tipo, stato):
    """
    Standardizza un DataFrame raw:
    - I file 'in gestione' hanno nome_regione/provincia/comune + NomeXxxValidato.
      Se NomeXxxValidato esiste già, droppiamo il duplicato nome_xxx.
    - Aggiunge colonne mancanti come NaN.
    - Aggiunge colonne 'tipo' e 'stato'.
    """
    df = df.copy()
    ALIAS = {
        'nome_regione':   'NomeRegioneValidato',
        'nome_provincia': 'NomeProvinciaValidato',
        'nome_comune':    'NomeComuneValidato',
    }
    for src, dst in ALIAS.items():
        if src in df.columns:
            if dst in df.columns:      # già presente la versione Validata
                df = df.drop(columns=[src])
            else:
                df = df.rename(columns={src: dst})
    for col in COLS_COMUNI + COLS_DESTINATI + COLS_GESTIONE:
        if col not in df.columns:
            df[col] = np.nan
    df['tipo']  = tipo
    df['stato'] = stato
    return df[ALL_COLS].reset_index(drop=True)

az_dest_p  = prepara_dataframe(az_dest,  'azienda',  'destinato')
az_gest_p  = prepara_dataframe(az_gest,  'azienda',  'in_gestione')
imm_dest_p = prepara_dataframe(imm_dest, 'immobile', 'destinato')
imm_gest_p = prepara_dataframe(imm_gest, 'immobile', 'in_gestione')

print('Colonne standardizzate:', ALL_COLS)

Colonne standardizzate: ['tipo', 'stato', 's_bene', 'genere', 'categoria', 'sottocategoria', 'distretto', 'NomeComuneValidato', 'NomeProvinciaValidato', 'NomeRegioneValidato', 'CODISTAT', 'CODISTATPROV', 'CODISTATREG', 'indirizzo', 'ufficio', 'procedura', 'tipologia', 'destinatario', 'fine', 'utilizzo', 'decreto_numero', 'decreto_anno', 'decreto_data', 'provvedimento', 'quota_confiscata', 'ufficio_giudiziario', 'tipologia_procedura']


## 3. Unificazione in un unico DataFrame

Concateniamo i quattro DataFrame. Ogni riga rappresenta un singolo bene confiscato: le colonne non applicabili (es. `decreto_anno` per i beni in gestione) saranno NaN.

In [4]:
df = pd.concat([az_dest_p, az_gest_p, imm_dest_p, imm_gest_p], ignore_index=True)
printInfo(df, 'Dataset Unificato — Beni Confiscati')


  Dataset Unificato — Beni Confiscati
  Righe: 36,301  |  Colonne: 27

  Colonna                        Tipo         Non-null   % null    Unici
  ----------------------------------------------------------------------
  tipo                           object          36301     0.0%        2
  stato                          object          36301     0.0%        2
  s_bene                         object          32563    10.3%    32528
  genere                         object          36301     0.0%        2
  categoria                      object          36301     0.0%       17
  sottocategoria                 object          35965     0.9%       49
  distretto                      object          36301     0.0%      171
  NomeComuneValidato             object          36216     0.2%     1797
  NomeProvinciaValidato          object          36289     0.0%      106
  NomeRegioneValidato            object          36289     0.0%       20
  CODISTAT                       float64         362

## 4. Pulizia dei dati

### 4.1 Normalizzazione codici ISTAT
Il codice ISTAT provincia (`CODISTATPROV`) deve essere una stringa di 3 cifre con zero-padding (es. `082` per Palermo). Analogamente, il codice comune (`CODISTAT`) deve essere a 6 cifre. Quando pandas carica un CSV con colonne numeriche, li converte in float aggiungendo `.0` — lo correggiamo.

In [5]:
df['CODISTATPROV'] = (
    df['CODISTATPROV']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.zfill(3)
)

df['CODISTAT'] = (
    df['CODISTAT']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.zfill(6)
)

print('Esempio CODISTATPROV:', df['CODISTATPROV'].value_counts().head().to_dict())
print('Esempio CODISTAT:', df['CODISTAT'].value_counts().head().to_dict())

Esempio CODISTATPROV: {'082': 6155, '080': 3359, '063': 2801, '081': 2180, '061': 1656}
Esempio CODISTAT: {'082053': 3376, '058091': 1071, '080063': 922, '063049': 690, '015146': 637}


### 4.2 Anno del decreto
`decreto_anno` è rilevante solo per i beni *destinati*. Lo convertiamo in intero nullable (`Int64`) così i NaN dei beni in gestione sono gestiti correttamente.

In [6]:
df['decreto_anno'] = pd.to_numeric(df['decreto_anno'], errors='coerce').astype('Int64')
print('Distribuzione anni decreto (ultimi 10 anni):')
print(df['decreto_anno'].value_counts().sort_index().tail(10))

Distribuzione anni decreto (ultimi 10 anni):
decreto_anno
2010     680
2011     249
2012     197
2013     460
2014     658
2015    1913
2016    1273
2017    2419
2018    1762
2019      12
Name: count, dtype: Int64


### 4.3 Quota confiscata
Nei file *in gestione*, il valore `0` per `quota_confiscata` significa 'dato non disponibile', non 'quota zero'. Lo sostituiamo con NaN.

In [7]:
mask_gest = df['stato'] == 'in_gestione'
df.loc[mask_gest & (df['quota_confiscata'] == 0), 'quota_confiscata'] = np.nan
print('Quota confiscata (beni in gestione) — statistiche:')
print(df.loc[mask_gest, 'quota_confiscata'].describe())

Quota confiscata (beni in gestione) — statistiche:
count    17525.000000
mean        91.534151
std         21.773767
min          1.000000
25%        100.000000
50%        100.000000
75%        100.000000
max        100.000000
Name: quota_confiscata, dtype: float64


### 4.4 Pulizia campi testuali
Rimuoviamo spazi superflui e normalizziamo i valori nulli nei campi stringa.

In [8]:
TEXT_COLS = [
    'NomeComuneValidato', 'NomeProvinciaValidato', 'NomeRegioneValidato',
    'categoria', 'sottocategoria', 'destinatario', 'fine', 'utilizzo',
    'provvedimento', 'tipologia_procedura'
]

for col in TEXT_COLS:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str).str.strip()
            .replace({'nan': np.nan, '': np.nan, 'None': np.nan})
        )

printInfo(df, 'Dopo pulizia testi')


  Dopo pulizia testi
  Righe: 36,301  |  Colonne: 27

  Colonna                        Tipo         Non-null   % null    Unici
  ----------------------------------------------------------------------
  tipo                           object          36301     0.0%        2
  stato                          object          36301     0.0%        2
  s_bene                         object          32563    10.3%    32528
  genere                         object          36301     0.0%        2
  categoria                      object          36301     0.0%       17
  sottocategoria                 object          35965     0.9%       49
  distretto                      object          36301     0.0%      171
  NomeComuneValidato             object          36216     0.2%     1797
  NomeProvinciaValidato          object          36289     0.0%      106
  NomeRegioneValidato            object          36289     0.0%       20
  CODISTAT                       object          36301     0.0%     1

  CODISTATPROV                   object          36301     0.0%      106
  CODISTATREG                    float64         36288     0.0%       20
  indirizzo                      object          15443    57.5%     8238
  ufficio                        object          15826    56.4%        5
  procedura                      object          15826    56.4%     1556
  tipologia                      object          15826    56.4%        8
  destinatario                   object          14730    59.4%       21
  fine                           object          12957    64.3%       11
  utilizzo                       object           5724    84.2%      813
  decreto_numero                 float64         15826    56.4%     5253
  decreto_anno                   Int64           15826    56.4%       41
  decreto_data                   object          15825    56.4%     1978
  provvedimento                  object          16663    54.1%       10
  quota_confiscata               float64         17

## 5. Arricchimento geografico: coordinate comunali da Wikidata

Per ogni bene, recuperiamo le **coordinate geografiche del comune** (P625 su Wikidata) tramite un join sul codice ISTAT a 6 cifre (`CODISTAT`).

Il mapping `codistat → wikidata_uri, lat, lon` è contenuto in `dataset/other/wikidata_comuni.csv`, ottenuto interrogando il SPARQL endpoint di Wikidata (vedere `dataset/querySparql.txt`). Questo approccio è completamente tracciabile e riproducibile: le coordinate provengono da una fonte LOD esterna (Wikidata, proprietà P625) e non da un dizionario hardcoded, in linea con i principi del progetto.

In [9]:
# Carica il mapping Wikidata: CODISTAT → wikidata_uri + lat + lon
wikidata = pd.read_csv(
    '../dataset/other/wikidata_comuni.csv',
    dtype={'codistat': str}
)

# Zero-padding a 6 cifre, coerente con la colonna CODISTAT del dataset
wikidata['codistat'] = wikidata['codistat'].str.zfill(6)

# Wikidata può avere coordinate multiple per lo stesso comune (es. coordinate storiche);
# teniamo solo la prima occorrenza per evitare righe duplicate nel merge.
wikidata = wikidata.drop_duplicates(subset='codistat', keep='first')

# Left join su CODISTAT: ogni bene riceve le coordinate del suo comune
df = df.merge(
    wikidata[['codistat', 'lat', 'lon']],
    left_on='CODISTAT',
    right_on='codistat',
    how='left'
).drop(columns=['codistat'])

assert len(df) == 36_301, f'Errore: righe dopo merge = {len(df)} (attese 36.301)'

n_con = df['lat'].notna().sum()
print(f'Beni con coordinate: {n_con:,} / {len(df):,} ({n_con/len(df)*100:.1f}%)')

# Comuni senza corrispondenza in Wikidata (P625 assente)
no_coords = df[df['lat'].isna()][['NomeComuneValidato', 'CODISTAT']].drop_duplicates()
if len(no_coords) > 0:
    print(f'Comuni senza coordinate Wikidata ({len(no_coords)}):')
    print(no_coords.to_string(index=False))
else:
    print('Tutti i comuni hanno coordinate Wikidata.')

Beni con coordinate: 35,744 / 36,301 (98.5%)
Comuni senza coordinate Wikidata (33):
    NomeComuneValidato CODISTAT
                   NaN   000nan
  OLEVANO SUL TUSCIANO   065082
 MONTECORVINO PUGLIANO   065072
             AGRIGENTO   000nan
              IGLESIAS   111035
                 OLBIA   090047
               SORBOLO   034037
                BUDONI   090091
               TEULADA   111089
                 NUORO   091051
          GOLFO ARANCI   090083
TAVARNELLE VAL DI PESA   048045
             ARZACHENA   090006
   GUIDONIA MONTECELIO   058047
        ZELO SURRIGONE   015246
 LOIRI PORTO SAN PAOLO   090084
               ALGHERO   090003
            TREVIGNANO   026085
          CASTELLABATE   065031
  SAN GIOVANNI SUERGIU   111063
           VALLE MOSSO   096073
              VERMEZZO   015235
                 MONTI   090041
  SANTA TERESA GALLURA   090063
              VILLAMAR   111097
                 PALAU   090054
                 QUART   007054
         VALTOURNENC

## 6. Riepilogo finale

In [10]:
printInfo(df, 'Dataset Finale — Beni Confiscati')

print('\n--- Distribuzione per tipo e stato ---')
print(df.groupby(['tipo', 'stato']).size().reset_index(name='n_beni').to_string(index=False))


  Dataset Finale — Beni Confiscati
  Righe: 36,301  |  Colonne: 29

  Colonna                        Tipo         Non-null   % null    Unici
  ----------------------------------------------------------------------
  tipo                           object          36301     0.0%        2
  stato                          object          36301     0.0%        2
  s_bene                         object          32563    10.3%    32528
  genere                         object          36301     0.0%        2
  categoria                      object          36301     0.0%       17
  sottocategoria                 object          35965     0.9%       49
  distretto                      object          36301     0.0%      171
  NomeComuneValidato             object          36216     0.2%     1797
  NomeProvinciaValidato          object          36289     0.0%      106
  NomeRegioneValidato            object          36289     0.0%       20
  CODISTAT                       object          36301 

## 7. Export

Esportiamo il dataset unificato e pulito in:
- **CSV** (`beni-confiscati.csv`) — formato principale
- **datapackage.json** — metadati strutturati secondo lo standard Frictionless Data

In [11]:
OUTPUT_PATH = '../dataset/processed/'
esporta_csv_e_frictionless(df, OUTPUT_PATH, filename='beni-confiscati')
print('Export completato.')

CSV salvato: ../dataset/processed/beni-confiscati.csv
datapackage.json salvato: ../dataset/processed/datapackage.json
Export completato.
